# Document Preparation - Bridge Pattern

Demonstrates converting a tree-parsed document to flat Markdown so that any
flat chunker (Fixed, Sentence) can still be applied.

```
Parser (mode="tree")  →  Document.root
        ↓  doc.to_markdown()         ← bridge
flat string (Markdown)
        ↓  FixedChunker / RecursiveChunker (string path)
List[Chunk]
```

**When to use this pattern**
- You need a flat chunker (e.g., FixedChunker) on a tree-parsed document.
- You want to inspect or post-process the full Markdown representation before chunking.

**Trade-off**
The `to_markdown()` conversion flattens the hierarchy. A `FixedChunker` applied
afterwards may cut in the middle of a section. `RecursiveChunker` applied to
the Markdown is more respectful of heading boundaries but still loses the
table-atomicity guarantee that the tree-walk path provides.

## Imports

In [ ]:
# Standard Library
import pathlib

# Third Party Library

# Private Library
from cleave.chunker.fixed import FixedChunker
from cleave.chunker.recursive import RecursiveChunker
from cleave.parsers.office.docx import DocxParser
from cleave.schemas import (
    ChunkParams,
    ContentBlock,
    ContentType,
    Document,
    DocumentPage,
    Source,
    SourceType,
)

## Fixture

In [ ]:
FIXTURES  = pathlib.Path.cwd().parent.parent.parent / "tests" / "fixtures"
DOCX_PATH = str(FIXTURES / "sample.docx")
print("DOCX:", DOCX_PATH)

## Step 1 — Parse as tree

In [ ]:
tree_doc = DocxParser(DOCX_PATH, mode="tree").parse()

print("root:", tree_doc.root.metadata.get("role"))
print("children:", len(tree_doc.root.children))

## Step 2 — Bridge: tree → Markdown string

In [ ]:
md_string = tree_doc.to_markdown()

print(f"Markdown length: {len(md_string)} chars\n")
print(md_string)

## Step 3 — Wrap Markdown in a flat Document
Any flat chunker expects `Document.pages`. We create a single virtual page from the
Markdown string. The source is borrowed from the original tree document.

In [ ]:
flat_page = DocumentPage(
    page_number=None,
    blocks=[ContentBlock(type=ContentType.text, content=md_string, position=0)],
)
flat_doc = Document(source=tree_doc.source, pages=[flat_page], total_pages=1)

print("flat_doc pages :", flat_doc.total_pages)
print("flat_doc chars :", len(flat_doc.full_text))

## FixedChunker on the flattened Markdown

In [ ]:
CHUNK_SIZE    = 200
CHUNK_OVERLAP = 20

fixed_chunks = FixedChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)).chunk(flat_doc)

print(f"FixedChunker  chunk_size={CHUNK_SIZE}  overlap={CHUNK_OVERLAP}  total={len(fixed_chunks)}\n")
for c in fixed_chunks:
    print(f"[{c.index}] chars {c.char_start:>4}–{c.char_end:<4}  tokens={c.token_count:>3}  │  {c.text[:70]!r}")

## RecursiveChunker on the flattened Markdown
Because Markdown heading delimiters (`\n# `, `\n## `) are in the string path's
delimiter hierarchy, the RecursiveChunker will respect heading boundaries more
often than FixedChunker — though table atomicity is no longer guaranteed.

In [ ]:
rec_chunks = RecursiveChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)).chunk(flat_doc)

print(f"RecursiveChunker (string path)  total={len(rec_chunks)}\n")
for c in rec_chunks:
    print(f"[{c.index}] tokens={c.token_count:>3}  │  {c.text[:80]!r}")

## Comparison: bridge vs. direct tree-walk

In [ ]:
from cleave.schemas import ContentType

# Direct tree-walk for reference
tree_chunks = RecursiveChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)).chunk(tree_doc)

print(f"{'Approach':<30}  {'Chunks':>6}  {'Avg chars':>9}  {'Table chunks':>12}")
print("-" * 65)

for label, chunks in [
    ("Bridge → FixedChunker",     fixed_chunks),
    ("Bridge → RecursiveChunker", rec_chunks),
    ("Direct tree-walk",           tree_chunks),
]:
    avg_chars    = sum(len(c.text) for c in chunks) / max(len(chunks), 1)
    table_chunks = sum(1 for c in chunks if c.content_type == ContentType.table)
    print(f"{label:<30}  {len(chunks):>6}  {avg_chars:>9.1f}  {table_chunks:>12}")

## Verdict

| Path | Table atomic? | Heading-aware? | Complexity |
|------|:---:|:---:|:---:|
| Bridge → FixedChunker | ✗ | ✗ | Low |
| Bridge → RecursiveChunker | ✗ | ✓ | Low |
| Direct tree-walk (RecursiveChunker) | ✓ | ✓ | None (just use the tree) |

Use the bridge pattern only when you need a specific flat chunker for compatibility
reasons. For production RAG, prefer the direct tree-walk path.